In [1]:
#데이터 처리
import pandas as pd

#MySQL 연결
import pymysql
from sqlalchemy import create_engine, text #연결 관리용으로만 가볍게
from pathlib import Path
from pprint import pprint

#.env 파일 활용하기
from dotenv import load_dotenv
import os
import time

from google.cloud import bigquery
import re

#env 파일 읽기
from google.cloud import bigquery

load_dotenv()

#SQLAlchemy의 engine을 미리 만들어 놓습니다.
DB_URL = f"mysql+pymysql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
engine = create_engine(DB_URL)

In [2]:
#구글 bigquery 연결을 만듭니다.
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../gcp-credentials.json'

project_id = os.getenv('GCP_PROJECT_ID')
dataset = os.getenv('BQ_DATASET_RC')
big_engine = create_engine(f'bigquery://{project_id}/{dataset}')

In [3]:
query = text("""
    select *
    from regional_climate
""")

with engine.connect() as conn:
    result = conn.execute(query)
    df_raw = pd.DataFrame(result.fetchall(), columns=result.keys())

In [4]:
query = text("""
    select *
    from regional_climate_meta
""")

with engine.connect() as conn:
    result = conn.execute(query)
    df_meta = pd.DataFrame(result.fetchall(), columns=result.keys())

In [5]:
#TODO
# 지점주소가 null로 되어있는 181지점 지점주소 직접 입력해주기
df_meta.loc[df_meta['지점명']=='서청주(예)','지점주소'] = '충청북도 청주시'

# 지점별 건수가 1건이고, 종료일이 2000년 이전인 폐관 관측소 제거
# 분석에 사용되지 않는 기간의 데이터
station_counts = df_meta.groupby('지점')['지점'].transform('count')
is_single = station_counts == 1
is_old_closed = df_meta['종료일'] <= '2000-12-31'

df_meta = df_meta[~(is_single & is_old_closed)]

# 주소에 '(산지)' 포함된 관측소 제거
df_meta = df_meta[~df_meta['지점주소'].str.contains('(산지)', na=False, regex=False)]

In [6]:
#컬럼에 있는 특수문자, 기호 제거
def clean_column_name(col):
    # 괄호 안 내용을 언더스코어로 변환: 평균기온(°C) → 평균기온_C
    col = re.sub(r'\(([^)]+)\)', r'_\1', col)
    # 허용되지 않는 특수문자 제거: °, /, . 등
    col = re.sub(r'[^\w가-힣]', '_', col)
    # 연속 언더스코어 정리
    col = re.sub(r'_+', '_', col)
    # 앞뒤 언더스코어 제거
    col = col.strip('_')
    return col

df_raw.columns = [clean_column_name(col) for col in df_raw.columns]
df_meta.columns = [clean_column_name(col) for col in df_meta.columns]

In [7]:
client = bigquery.Client()
table_id = f'{project_id}.{dataset}.raw'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE'  # 기존 데이터 날리고 새로 넣기
)

job = client.load_table_from_dataframe(df_raw, table_id, job_config=job_config)
job.result()
print(f'{job.output_rows}행 입력 완료')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


27668행 입력 완료


In [8]:
client = bigquery.Client()
table_id = f'{project_id}.{dataset}.meta'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE'  # 기존 데이터 날리고 새로 넣기
)

job = client.load_table_from_dataframe(df_meta, table_id, job_config=job_config)
job.result()
print(f'{job.output_rows}행 입력 완료')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


140행 입력 완료
